# Transmission Expansion Planning

Transmission expansion planning lets the model decide whether to build candidate lines. A line is a candidate when it has a positive investment cost in `oT_Data_Network`. The 9-node case has one candidate line, between Node_1 and Node_4 (circuit `dc1`).

We turn the build decision into a binary one and let the model choose.

## 1. Set up a working copy of the case

In [1]:
import os, shutil
import pandas as pd

DIR = "work_TEP"          # parent folder that will hold the case
CaseName = "9n"      # we reuse the 9-node case from notebook 01

if os.path.exists(DIR):
    shutil.rmtree(DIR)
shutil.copytree(CaseName, os.path.join(DIR, CaseName))

# A coarse time resolution keeps the run fast for this tutorial.
param = os.path.join(DIR, CaseName, "oT_Data_Parameter_9n.csv")
df = pd.read_csv(param)
df.loc[:, "TimeStep"] = 24
df.to_csv(param, index=False)
print("Working copy of the 9n case is ready in", DIR)

Working copy of the 9n case is ready in work_TEP


## 2. Activate binary network investment

Set `IndBinNetInvest = 1` in `oT_Data_Option` (binary build decision) and ignore generation investment. Then mark the candidate line with `BinaryInvestment = 'Yes'` in `oT_Data_Network`.

In [2]:
opt = pd.read_csv(os.path.join(DIR, CaseName, "oT_Data_Option_9n.csv"))
opt.loc[0, "IndBinNetInvest"] = 1
opt.loc[0, ["IndBinGenInvest", "IndBinGenRetirement"]] = 2
opt.to_csv(os.path.join(DIR, CaseName, "oT_Data_Option_9n.csv"), index=False)

net = pd.read_csv(os.path.join(DIR, CaseName, "oT_Data_Network_9n.csv"))
cand = (net["InitialNode"] == "Node_1") & (net["FinalNode"] == "Node_4") & (net["Circuit"] == "dc1")
net["BinaryInvestment"] = net["BinaryInvestment"].astype(object)   # let the column hold the Yes/No flag
net.loc[cand, "BinaryInvestment"] = "Yes"
net.to_csv(os.path.join(DIR, CaseName, "oT_Data_Network_9n.csv"), index=False)
net.loc[cand, ["InitialNode", "FinalNode", "Circuit", "FixedInvestmentCost", "BinaryInvestment"]]

,InitialNode,FinalNode,Circuit,FixedInvestmentCost,BinaryInvestment
12,Node_1,Node_4,dc1,100.0,Yes


## 3. Run the model

In [3]:
from openTEPES.openTEPES import openTEPES_run

model = openTEPES_run(DIR, CaseName, "appsi_highs", "Yes", "No")
print("Total system cost [MEUR]:", round(model.vTotalSCost(), 3))

Input data                             ****
Reading the CSV files                  ...  0 s
Reading    input data                  ...  0 s


Setting up input data                  ...  1 s
Setting up variables                   ...  0 s
Total cost o.f.      model formulation ****
Investment elec      model formulation ****
Period 2030, Scenario sc01, Stage st1
Generation oper o.f. model formulation ****


Investment & operation var constraints ****
Inertia, oper resr, demand constraints ****
Storage   scheduling       constraints ****
Unit commitment            constraints ****


Ramp and min up/down time  constraints ****
Network    switching model constraints ****
Network    operation model constraints ****
Problem solving                        #### 1


Termination condition:  optimal
Problem solving with fixed investments #### 1


  Total system                 cost [MEUR]  161.60310205056308  Constraints 41136  Variables 50966  Seconds 7
***** Period: 2030, Scenario: sc01, Stage: st1 ******
  Total generation  investment cost [MEUR]  0
  Total generation  retirement cost [MEUR]  0
  Total reservoir   investment cost [MEUR]  0.0
  Total network     investment cost [MEUR]  7.5
  Total H2   pipe   investment cost [MEUR]  0.0
  Total heat pipe   investment cost [MEUR]  0.0
  Total generation  operation  cost [MEUR]  154.09859358512946
  Total consumption operation  cost [MEUR]  0.0028953504033765167
  Total emission               cost [MEUR]  0.0
  Total network losses penalty cost [MEUR]  0.0016131150311659611
  Total reliability electr     cost [MEUR]  0.0
Writing            investment results  ...  0 s
Writing          cost summary results  ...  0 s
Writing           KPI summary results  ...  0 s
Writing elect network summary results  ...  0 s
Writing           reliability indexes  ...  0 s


Writing           flexibility results  ...  0 s
Writing  generation operation results  ...  0 s


/private/tmp/claude-501/-Users-philias-ai-research-repos-openTEPES-tutorial/8f6d4ac5-9ff1-45bc-8a3c-5cc562abe3f1/scratchpad/venv312/lib/python3.12/site-packages/altair/utils/core.py:264: UserWarning: I don't know how to infer vegalite type from 'empty'.  Defaulting to nominal.
  warnings.warn(


Writing         ESS operation results  ...  0 s
Writing elect netwk operation results  ...  0 s
Writing  marginal information results  ...  0 s


Writing              economic results  ...  0 s
Plotting electricity network     maps  ...  0 s
Total system cost [MEUR]: 161.603


## 4. Did the line get built?

The network investment decision is written to `oT_Result_NetworkInvestment`. A value of 1 means the candidate line is built.

In [4]:
inv = pd.read_csv(os.path.join(DIR, CaseName, "oT_Result_NetworkInvestmentPerUnit_9n.csv"))
inv

,Period,Node_1
0,NaN,Node_4
1,NaN,dc1
2,2030.0,1.0
